# Notebook 01: Run Model (GPU Execution)

**Project:** Confidence-Based Abstention for LLM ICD-10 Coding (Paper C1)

This notebook runs on Google Colab Pro with a GPU runtime (T4, L4, or A100).
It mounts Google Drive for output persistence, clones the repository, checks prerequisites,
executes diagnostics (Gate A), runs a 20-note smoke benchmark, and performs the full
resumable generation across all four stages (`greedy`, `samples`, `s2`, `s4`).

> **Important:** Run one session per model (`MODEL = "llama3"` first, then restart runtime and run `MODEL = "biomistral"`).
> All long stages are fully resumable.

In [ ]:
# Cell 1 – GPU check
!nvidia-smi

In [ ]:
# Cell 2 – Drive (outputs persist here)
from google.colab import drive
drive.mount('/content/drive')
RUN = "/content/drive/MyDrive/c1_runs/c1_run01"      # same path for both models

In [ ]:
# Cell 3 – code + dependencies (private repo: put a GitHub token in Colab Secrets as GH_TOKEN)
from google.colab import userdata
gh = userdata.get("GH_TOKEN")
!git clone https://{gh}@github.com/<you>/<repo>.git /content/c1 || (cd /content/c1 && git pull)
%cd /content/c1
!pip -q install -r requirements-colab.txt

In [ ]:
# Cell 4 – secrets and cache
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HOME"] = "/content/drive/MyDrive/hf_cache"   # optional: skips re-downloading the model next session

In [ ]:
# Cell 5 – corpus + split (first session only; harmless to rerun)
!python -m c1.cli make-corpus --config configs/c1.yaml --out $RUN
!python -m c1.cli split --config configs/c1.yaml --out $RUN

In [ ]:
# Cell 6 – checks (Gate A): GPU/library versions, determinism, S3 alignment, S4 Yes/No token
MODEL = "llama3"      # or "biomistral"
!python -m c1.cli selfcheck --config configs/c1.yaml --out $RUN --model $MODEL
!python -m c1.cli estimate  --config configs/c1.yaml --out $RUN --model $MODEL

In [ ]:
# Cell 7 – smoke run (20 dev notes, separate output folder)
!python -m c1.cli run --config configs/c1.yaml --out {RUN}_smoke --model $MODEL --stages greedy,samples,s2,s4 --split dev --limit 20
!python -m c1.cli build-codes --config configs/c1.yaml --out {RUN}_smoke --model $MODEL

In [ ]:
# Cell 8 – FULL run (only after the smoke run looks right)
!python -m c1.cli run --config configs/c1.yaml --out $RUN --model $MODEL --stages greedy,samples,s2,s4

In [ ]:
# Cell 9 – verify completeness
!python -m c1.cli sanity --config configs/c1.yaml --out $RUN --stage-counts-only